In [ ]:
#库包安装以及停用词文档下载
%pip install pkuseg tqdm pandas scikit-learn
%wget https://raw.githubusercontent.com/goto456/stopwords/master/cn_stopwords.txt -O stopwords.txt


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
--2025-11-03 07:03:48--  https://raw.githubusercontent.com/goto456/stopwords/master/cn_stopwords.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 100.65.138.175
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|100.65.138.175|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4717 (4.6K) [text/plain]
Saving to: ‘stopwords.txt’

stopwords.txt       100%[===================>]   4.61K  --.-KB/s    in 0s      

2025-11-03 07:03:48 (48.4 MB/s) - ‘stopwords.txt’ saved [4717/4717]



In [ ]:
%pip install jieba

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 30.6 MB/s  0:00:00 eta 0:00:01
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'jieba' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'jieba'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for jieba: filename=jieba-0.42.1-py3-none-any.whl size=19314509 sha256=62533c72e85257c53ef4f9284ca0f7120aed0b55734e2a5f56bd33fc693753ba
  Stored in directory: /root/.cache/pip/wheels/2f/f0/17/62f6fadd3e378fe5ad9beb890f4ae81a112484dc0814fa1d81
Successfully built jieba


In [ ]:
#给予jieba的字典分词
import os
import re
import json
import jieba
from tqdm import tqdm

# 路径配置
DATASET_DIR = "CSTS"               
OUT_DIR = "CSTS_jieba"      
os.makedirs(OUT_DIR, exist_ok=True)

# 仅处理的相似性子集 
TARGET_SETS = [
    "LCQMC",
    "AFQMC",
    "OPPO-xiaobu",
]

# 停用词加载
STOPWORDS = set()
if os.path.exists("stopwords.txt"):
    with open("stopwords.txt", "r", encoding="utf-8") as f:
        STOPWORDS = {w.strip() for w in f if w.strip()}

# 文本清洗 + 分词 
def clean_and_cut(text):
    """清洗 + jieba分词 + 去停用词"""
    text = str(text).strip()
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[^\u4e00-\u9fa5a-zA-Z0-9]", "", text) # 去除非中文、英文、数字字符
    words = jieba.cut(text)
    if STOPWORDS:
        words = [w for w in words if w not in STOPWORDS]
    return " ".join(words)

# 处理单个文件
def process_txt_file(in_file, out_file):
    """读取三列制表符数据: sentence1, sentence2, label"""
    data = []
    with open(in_file, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) != 3:
                continue
            s1, s2, label = parts
            try:
                if "." in label:
                    label_value = float(label)
                else:
                    label_value = int(label)
            except ValueError:
                continue
            s1 = clean_and_cut(s1)
            s2 = clean_and_cut(s2)
            data.append({"sentence1": s1, "sentence2": s2, "label": label_value})

    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    with open(out_file, "w", encoding="utf-8") as f:
        for d in data:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

    print(f"✅ {in_file} -> {len(data)} 条样本")

# 处理整个数据集
def process_dataset(name):
    src_dir = os.path.join(DATASET_DIR, name)
    if not os.path.isdir(src_dir):
        return
    print(f"\n==== 处理数据集：{name} ====")
    dst_dir = os.path.join(OUT_DIR, name)
    os.makedirs(dst_dir, exist_ok=True)

    total = 0
    for split in ["train", "dev", "test"]:
        in_path = os.path.join(src_dir, f"{split}.txt")
        out_path = os.path.join(dst_dir, f"{split}.jsonl")
        if os.path.exists(in_path):
            process_txt_file(in_path, out_path)
            total += 1
        else:
            print(f"⚠️ 未找到 {in_path}")
    if total == 0:
        print("⚠️ 该子数据集未生成任何文件")


subsets = [d for d in os.listdir(DATASET_DIR)
           if os.path.isdir(os.path.join(DATASET_DIR, d)) and d in TARGET_SETS]

print(f"jieba将处理以下相似性数据集：{subsets}")

for name in subsets:
    process_dataset(name)

print("\n✅ 全部相似性数据集预处理完成，输出目录:", OUT_DIR)


jieba将处理以下相似性数据集：['OPPO-xiaobu', 'LCQMC', 'AFQMC']

==== 处理数据集：OPPO-xiaobu ====
✅ CSTS/OPPO-xiaobu/train.txt -> 167168 条样本
✅ CSTS/OPPO-xiaobu/dev.txt -> 10000 条样本
✅ CSTS/OPPO-xiaobu/test.txt -> 0 条样本

==== 处理数据集：LCQMC ====
✅ CSTS/LCQMC/train.txt -> 238766 条样本
✅ CSTS/LCQMC/dev.txt -> 8802 条样本
✅ CSTS/LCQMC/test.txt -> 12500 条样本

==== 处理数据集：AFQMC ====
✅ CSTS/AFQMC/train.txt -> 34334 条样本
✅ CSTS/AFQMC/dev.txt -> 4316 条样本
✅ CSTS/AFQMC/test.txt -> 0 条样本

✅ 全部相似性数据集预处理完成，输出目录: CSTS_jieba


In [ ]:
# === Jieba + TF-IDF 基线评估 ===
import os, json, numpy as np
from typing import List, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

DATA_DIR = "CSTS_jieba"
SUBSETS = ["AFQMC", "LCQMC", "OPPO-xiaobu"]

# 数据加载
def load_jsonl(path: str) -> Tuple[List[str], List[str], List[int]]:
    s1, s2, y = [], [], []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            try:
                d = json.loads(line)
                a = str(d.get("sentence1", "")).strip()
                b = str(d.get("sentence2", "")).strip()
                lab = d.get("label", None)
                if not a or not b or lab is None:
                    continue
                lab = int(float(lab))
                if lab not in [0, 1]:
                    continue
                s1.append(a)
                s2.append(b)
                y.append(lab)
            except Exception as e:
                print(f"⚠️ 跳过 {os.path.basename(path)} 第 {i} 行: {e}")
                continue

    n = min(len(s1), len(s2), len(y))
    if len(s1) != len(s2) or len(s1) != len(y):
        print(f"⚠️ {os.path.basename(path)} 样本数量不齐: s1={len(s1)}, s2={len(s2)}, y={len(y)} → 截断为 {n}")
    return s1[:n], s2[:n], y[:n]

# 阈值搜索
def best_threshold(y_true: np.ndarray, sims: np.ndarray) -> float:
    """在 train 上自动扫描相似度阈值以最大化 F1"""
    cands = np.unique(np.round(sims, 4))
    if 0.5 not in cands:
        cands = np.append(cands, 0.5)
    best_t, best_f1 = 0.5, -1.0
    for t in cands:
        preds = (sims >= t).astype(int)
        f1 = f1_score(y_true, preds)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

# 相似度计算
def compute_sims(X1, X2, batch_size=10000):
    """分批计算每对句子的余弦相似度"""
    sims = []
    for i in range(0, X1.shape[0], batch_size):
        X1b = X1[i:i+batch_size]
        X2b = X2[i:i+batch_size]

        sims.extend(X1b.multiply(X2b).sum(axis=1).A1)
    return np.array(sims)

# split 评估
def evaluate_split(vec, s1, s2, y, thres, dense_fallback=True):
    """评估单个 split（train/dev/test）"""
    n = min(len(s1), len(s2), len(y))
    s1, s2, y = s1[:n], s2[:n], y[:n]

    # 分开 transform，保证 TF-IDF 维度一致
    X1 = vec.transform(s1)
    X2 = vec.transform(s2)

    try:
        sims = compute_sims(X1, X2)
    except Exception as e:
        if dense_fallback:
            print(f"⚠️ 稀疏计算失败，自动切换稠密模式: {e}")
            sims = compute_sims(X1, X2)
        else:
            raise e

    preds = (sims >= thres).astype(int)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds)
    return acc, f1

# 主逻辑
def run_one_subset(subset: str):
    print(f"\n==== TF-IDF （jieba）：{subset} ====")
    base = os.path.join(DATA_DIR, subset)
    paths = {split: os.path.join(base, f"{split}.jsonl") for split in ["train", "dev", "test"]}
    exists = {k: os.path.exists(v) for k, v in paths.items()}

    if not any(exists.values()):
        print("  ⚠️ 未找到任何 split 的 jsonl，跳过。")
        return

    # 构建词表
    fit_corpus = []
    for split in ["train", "test"]:
        if exists[split]:
            s1, s2, _ = load_jsonl(paths[split])
            fit_corpus.extend(s1 + s2)
    vec = TfidfVectorizer(
        token_pattern=r"[^ ]+",
        lowercase=False,
        min_df=2,
        max_df=0.95,
        norm="l2"  # 确保点积 = 余弦
    )
    vec.fit(fit_corpus)
    print(f"  ✔️ 构建 TF-IDF 词表：{len(fit_corpus)} 条文本，特征数={len(vec.get_feature_names_out())}")

    # 阈值选择

    tr_s1, tr_s2, tr_y = load_jsonl(paths["train"])
    X1 = vec.transform(tr_s1)
    X2 = vec.transform(tr_s2)
    sims_train = compute_sims(X1, X2)
    pos_mean = np.mean(sims_train[np.array(tr_y) == 1])
    neg_mean = np.mean(sims_train[np.array(tr_y) == 0])
    thres = (pos_mean + neg_mean) / 2
    print(f"  🔎 在 train 上估计阈值：{thres:.4f}")
    

    # 评估

    s1, s2, y = load_jsonl(paths["dev"])
    acc, f1 = evaluate_split(vec, s1, s2, y, thres)
    print(f"  ✅ dev: ACC={acc:.4f}  F1={f1:.4f}  N={len(y)}")


def main():
    for subset in SUBSETS:
        run_one_subset(subset)

if __name__ == "__main__":
    main()



==== TF-IDF （jieba）：AFQMC ====
  ✔️ 构建 TF-IDF 词表：68458 条文本，特征数=3983
  🔎 在 train 上估计阈值：0.3209
  ✅ dev: ACC=0.5557  F1=0.4099  N=4303

==== TF-IDF （jieba）：LCQMC ====
  ✔️ 构建 TF-IDF 词表：502408 条文本，特征数=36137
  🔎 在 train 上估计阈值：0.7020
  ✅ dev: ACC=0.6599  F1=0.6861  N=8801

==== TF-IDF （jieba）：OPPO-xiaobu ====
  ✔️ 构建 TF-IDF 词表：325982 条文本，特征数=21040
  🔎 在 train 上估计阈值：0.4719
  ✅ dev: ACC=0.6308  F1=0.5137  N=9753


In [ ]:
#更换分词方法为pkuseg
import os, re, json, csv
from glob import glob
from tqdm import tqdm


DATASET_DIR = "CSTS"
OUT_DIR = "CSTS_pkuseg"   
os.makedirs(OUT_DIR, exist_ok=True)


TARGET_SETS = ["AFQMC", "LCQMC", "OPPO-xiaobu"]


USE_TOKENIZE = True
STOPWORDS = set()
if os.path.exists("stopwords.txt"):
    with open("stopwords.txt", "r", encoding="utf-8") as f:
        STOPWORDS = {w.strip() for w in f if w.strip()}

import pkuseg
seg = pkuseg.pkuseg(model_name="default") if USE_TOKENIZE else None



def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\u4e00-\u9fa5a-zA-Z0-9 ]", "", text)
    return text.strip()

def tokenize(text):
    if not USE_TOKENIZE or seg is None:
        return clean_text(text)
    words = seg.cut(clean_text(text))
    if STOPWORDS:
        words = [w for w in words if w not in STOPWORDS]
    return " ".join(words)


def load_txt(path):
    """读取 train.txt/dev.txt/test.txt 类型文件"""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]
    has_header = lines[0].count("\t") == 2 and any(
        x in lines[0].lower() for x in ["sentence", "text", "label"]
    )
    if has_header:
        reader = csv.DictReader(lines, delimiter="\t")
        for row in reader:
            data.append(row)
    else:
        for line in lines:
            parts = line.split("\t")
            if len(parts) >= 3:
                s1, s2, label = parts[0], parts[1], parts[2]
                data.append({"sentence1": s1, "sentence2": s2, "label": label})
    return data

def save_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 分割文件处理
def process_split(in_file, out_file):
    data = load_txt(in_file)
    if not data:
        return 0
    processed = []
    for d in data:
        s1 = tokenize(d.get("sentence1", ""))
        s2 = tokenize(d.get("sentence2", ""))
        label = d.get("label", None)
        if "." in label:
            label = float(label)
        else:
            label= int(label)
        processed.append({"sentence1": s1, "sentence2": s2, "label": label})
    save_jsonl(out_file, processed)
    return len(processed)

# 主流程
subs = [d for d in os.listdir(DATASET_DIR)
        if os.path.isdir(os.path.join(DATASET_DIR, d)) and d in TARGET_SETS]

print(f"pkuseg将处理以下相似性数据集：{subsets}")

for ds in subs:
    ds_dir = os.path.join(DATASET_DIR, ds)
    print(f"\n==== 处理数据集：{ds} ====")
    out_dir = os.path.join(OUT_DIR, ds)
    os.makedirs(out_dir, exist_ok=True)
    total = 0
    for split in ["train", "dev", "test"]:
        in_file = os.path.join(ds_dir, f"{split}.txt")
        if not os.path.exists(in_file):
            print(f" - 未找到 {split}.txt")
            continue
        out_file = os.path.join(out_dir, f"{split}.jsonl")
        
        n = process_split(in_file, out_file)
        print(f"✅ {in_file} -> {n} 条样本")
        total += n
    if total == 0:
        print(" ⚠️ 该子数据集未生成任何样本")

print("\n✅ 全部完成，输出目录:", OUT_DIR)


pkuseg将处理以下相似性数据集：['OPPO-xiaobu', 'LCQMC', 'AFQMC']

==== 处理数据集：OPPO-xiaobu ====
✅ CSTS/OPPO-xiaobu/train.txt -> 167168 条样本
✅ CSTS/OPPO-xiaobu/dev.txt -> 10000 条样本
✅ CSTS/OPPO-xiaobu/test.txt -> 0 条样本

==== 处理数据集：LCQMC ====
✅ CSTS/LCQMC/train.txt -> 238766 条样本
✅ CSTS/LCQMC/dev.txt -> 8802 条样本
✅ CSTS/LCQMC/test.txt -> 12500 条样本

==== 处理数据集：AFQMC ====
✅ CSTS/AFQMC/train.txt -> 34334 条样本
✅ CSTS/AFQMC/dev.txt -> 4316 条样本
✅ CSTS/AFQMC/test.txt -> 0 条样本

✅ 全部完成，输出目录: CSTS_pkuseg


In [ ]:
# === pkuseg + TF-IDF 基线评估 ===
import os, json, numpy as np
from typing import List, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

DATA_DIR = "CSTS_pkuseg"
SUBSETS = ["AFQMC", "LCQMC", "OPPO-xiaobu"]

# 数据加载
def load_jsonl(path: str) -> Tuple[List[str], List[str], List[int]]:
    s1, s2, y = [], [], []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            try:
                d = json.loads(line)
                a = str(d.get("sentence1", "")).strip()
                b = str(d.get("sentence2", "")).strip()
                lab = d.get("label", None)
                if not a or not b or lab is None:
                    continue
                lab = int(float(lab))
                if lab not in [0, 1]:
                    continue
                s1.append(a)
                s2.append(b)
                y.append(lab)
            except Exception as e:
                print(f"⚠️ 跳过 {os.path.basename(path)} 第 {i} 行: {e}")
                continue

    n = min(len(s1), len(s2), len(y))
    if len(s1) != len(s2) or len(s1) != len(y):
        print(f"⚠️ {os.path.basename(path)} 样本数量不齐: s1={len(s1)}, s2={len(s2)}, y={len(y)} → 截断为 {n}")
    return s1[:n], s2[:n], y[:n]

# 阈值搜索
def best_threshold(y_true: np.ndarray, sims: np.ndarray) -> float:
    """在 train 上自动扫描相似度阈值以最大化 F1"""
    cands = np.unique(np.round(sims, 4))
    if 0.5 not in cands:
        cands = np.append(cands, 0.5)
    best_t, best_f1 = 0.5, -1.0
    for t in cands:
        preds = (sims >= t).astype(int)
        f1 = f1_score(y_true, preds)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

# 相似度计算
def compute_sims(X1, X2, batch_size=10000):
    """分批计算每对句子的余弦相似度"""
    sims = []
    for i in range(0, X1.shape[0], batch_size):
        X1b = X1[i:i+batch_size]
        X2b = X2[i:i+batch_size]
        sims.extend(X1b.multiply(X2b).sum(axis=1).A1)
    return np.array(sims)

# split 评估
def evaluate_split(vec, s1, s2, y, thres, dense_fallback=True):
    """评估单个 split（train/dev/test）"""
    n = min(len(s1), len(s2), len(y))
    s1, s2, y = s1[:n], s2[:n], y[:n]

    # 分开 transform，保证 TF-IDF 维度一致
    X1 = vec.transform(s1)
    X2 = vec.transform(s2)

    try:
        sims = compute_sims(X1, X2, dense=False)
    except Exception as e:
        if dense_fallback:
            print(f"⚠️ 稀疏计算失败，自动切换稠密模式: {e}")
            sims = compute_sims(X1, X2, dense=True)
        else:
            raise e

    preds = (sims >= thres).astype(int)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds)
    return acc, f1


def run_one_subset(subset: str):
    print(f"\n==== TF-IDF 评测（pkuseg）：{subset} ====")
    base = os.path.join(DATA_DIR, subset)
    paths = {split: os.path.join(base, f"{split}.jsonl") for split in ["train", "dev", "test"]}
    exists = {k: os.path.exists(v) for k, v in paths.items()}

    if not any(exists.values()):
        print("  ⚠️ 未找到任何 split 的 jsonl，跳过。")
        return

    # 构建词表
    fit_corpus = []
    for split in ["train", "test"]:
        if exists[split]:
            s1, s2, _ = load_jsonl(paths[split])
            fit_corpus.extend(s1 + s2)
    vec = TfidfVectorizer(
        token_pattern=r"[^ ]+",
        lowercase=False,
        min_df=2,
        max_df=0.95,
        norm="l2"  # 确保点积 = 余弦
    )
    vec.fit(fit_corpus)
    print(f"  ✔️ 构建 TF-IDF 词表：{len(fit_corpus)} 条文本，特征数={len(vec.get_feature_names_out())}")

    # 阈值选择

    tr_s1, tr_s2, tr_y = load_jsonl(paths["train"])
    X1 = vec.transform(tr_s1)
    X2 = vec.transform(tr_s2)
    sims_train = compute_sims(X1, X2)
    pos_mean = np.mean(sims_train[np.array(tr_y) == 1])
    neg_mean = np.mean(sims_train[np.array(tr_y) == 0])
    thres = (pos_mean + neg_mean) / 2
    print(f"  🔎 在 train 上估计阈值：{thres:.4f}")
    



    s1, s2, y = load_jsonl(paths["dev"])
    acc, f1 = evaluate_split(vec, s1, s2, y, thres)
    print(f"  ✅ dev: ACC={acc:.4f}  F1={f1:.4f}  N={len(y)}")


def main():
    for subset in SUBSETS:
        run_one_subset(subset)

if __name__ == "__main__":
    main()



==== TF-IDF 评测（pkuseg）：AFQMC ====
  ✔️ 构建 TF-IDF 词表：68238 条文本，特征数=3503
  🔎 在 train 上估计阈值：0.3286
  ✅ dev: ACC=0.5474  F1=0.4102  N=4282

==== TF-IDF 评测（pkuseg）：LCQMC ====
  ✔️ 构建 TF-IDF 词表：502390 条文本，特征数=38691
  🔎 在 train 上估计阈值：0.6706
  ✅ dev: ACC=0.6647  F1=0.6938  N=8801

==== TF-IDF 评测（pkuseg）：OPPO-xiaobu ====
  ✔️ 构建 TF-IDF 词表：325272 条文本，特征数=19672
  🔎 在 train 上估计阈值：0.4971
  ✅ dev: ACC=0.6386  F1=0.5266  N=9727
